In [1]:
!pip install sympy --upgrade -q
!pip install torchvision --upgrade -q

In [2]:
import os, torch, gc
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import models, transforms
from sklearn.metrics import classification_report, f1_score

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128
CUDA: True


In [3]:
PROJECT_DIR     = '/content/drive/MyDrive/DocFusion'
FINDITAGAIN_DIR = f'{PROJECT_DIR}/data/findit2'
MODELS_DIR      = '/content/model'

os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:

for item in os.listdir(FINDITAGAIN_DIR):
    item_path = f'{FINDITAGAIN_DIR}/{item}'
    if os.path.isdir(item_path):
        files = os.listdir(item_path)
        imgs  = [f for f in files if f.endswith('.png')]
        txts  = [f for f in files if f.endswith('.txt')]
        print(f"  {item}/  → {len(imgs)} images, {len(txts)} txt files")
        if imgs:
            print(f"    Sample: {imgs[0]}")
    else:
        print(f"  {item}")

In [ ]:

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH   = f'{PROJECT_DIR}/data/findit2'
BATCH_SIZE  = 16
EPOCHS      = 10
LR          = 1e-4
print(f"✅ Using: {DEVICE}")


data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_transforms['train'] = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class ForgeryDataset(Dataset):
    def __init__(self, df, split, transform=None):
        self.df        = df
        self.split     = split
        self.transform = transform
        self.root_dir  = os.path.join(BASE_PATH, split)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['image']
        img_path = os.path.join(self.root_dir, img_name)
        image    = Image.open(img_path).convert('RGB')
        label    = int(self.df.iloc[idx]['forged'])
        if self.transform:
            image = self.transform(image)
        return image, label

def filter_missing(df, split):
    valid_rows = []
    for _, row in df.iterrows():
        img_path = f'{BASE_PATH}/{split}/{row["image"]}'
        if os.path.exists(img_path):
            valid_rows.append(row)
    filtered_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    print(f"✅ {split}: {len(df)} → {len(filtered_df)} (removed {len(df)-len(filtered_df)} missing)")
    return filtered_df


train_df = filter_missing(pd.read_csv(os.path.join(BASE_PATH, 'train.txt')), 'train')
val_df   = filter_missing(pd.read_csv(os.path.join(BASE_PATH, 'val.txt')),   'val')
test_df  = filter_missing(pd.read_csv(os.path.join(BASE_PATH, 'test.txt')),  'test')


df_genuine        = train_df[train_df['forged'] == 0]
df_forged         = train_df[train_df['forged'] == 1]
df_forged_up      = df_forged.sample(len(df_genuine), replace=True, random_state=42)
train_df_balanced = pd.concat([df_genuine, df_forged_up]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Balanced train: {len(train_df_balanced)} samples")
print(f"   Genuine: {(train_df_balanced['forged']==0).sum()}")
print(f"   Forged:  {(train_df_balanced['forged']==1).sum()}")


train_ds     = ForgeryDataset(train_df_balanced, 'train', data_transforms['train'])
val_ds       = ForgeryDataset(val_df,            'val',   data_transforms['val'])
test_ds      = ForgeryDataset(test_df,           'test',  data_transforms['val'])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTrain loader: {len(train_loader)} batches")
print(f"Val loader:   {len(val_loader)} batches")
print(f"Test loader:  {len(test_loader)} batches")

In [ ]:
import torch.optim as optim

model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(1280, 2)

for param in model.parameters():
    param.requires_grad = False

for param in model.features[-2:].parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True




model = model.to(DEVICE)

print(f"EfficientNet-B0 loaded!")
print(f"Device: {DEVICE}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)


images, labels = next(iter(train_loader))
images, labels = images.to(DEVICE), labels.to(DEVICE)

with torch.no_grad():
    outputs = model(images)
    loss    = criterion(outputs, labels)

print("Ready!")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print(f"Forward pass works!")
print(f"Loss:   {loss.item():.4f}")
print(f"Output: {outputs.shape}")

In [22]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    f1        = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    acc       = accuracy_score(all_labels, all_preds)
    recall    = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    return total_loss / len(loader), acc, precision, recall, f1

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    f1        = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    acc       = accuracy_score(all_labels, all_preds)
    recall    = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    return total_loss / len(loader), acc, precision, recall, f1

In [ ]:
best_f1 = 0
for epoch in range(EPOCHS):
    train_loss, train_acc, train_p, train_r, train_f1 = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc,   val_p,   val_r,   val_f1   = val_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} P: {train_p:.4f} R: {train_r:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f} P: {val_p:.4f} R: {val_r:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f'{MODELS_DIR}/efficientnet_best.pth')
        print(f"Best model saved! F1: {best_f1:.4f}")

print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")

In [ ]:
import torch
from huggingface_hub import HfApi, upload_file
import os


model.load_state_dict(torch.load(f'{MODELS_DIR}/efficientnet_best.pth'))
model = model.to('cpu')


torch.save(model.state_dict(), f'{MODELS_DIR}/efficientnet_best.pth')


api      = HfApi()
repo_id  = "Zakariya007/docfusion-v2"


api.create_repo(repo_id=repo_id, exist_ok=True)


api.upload_file(
    path_or_fileobj = f'{MODELS_DIR}/efficientnet_best.pth',
    path_in_repo    = "efficientnet_best.pth",
    repo_id         = repo_id,
)

print("✅ EfficientNet pushed to HF Hub!")
print(f"   https://huggingface.co/{repo_id}")

In [ ]:
import json


config = {
    "model_type":   "efficientnet_b0",
    "num_classes":  2,
    "input_size":   224,
    "classes":      {0: "genuine", 1: "forged"},
    "threshold":    0.5
}

with open('/tmp/config.json', 'w') as f:
    json.dump(config, f)

api.upload_file(
    path_or_fileobj = '/tmp/config.json',
    path_in_repo    = 'config.json',
    repo_id         = "Zakariya007/docfusion-v2"
)


In [ ]:
from torchvision import transforms
from PIL import Image
import torch


model.load_state_dict(torch.load(f'{MODELS_DIR}/efficientnet_best.pth'))
model = model.to(DEVICE)
model.eval()


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


genuine_row = val_df[val_df['forged'] == 0].iloc[0]
forged_row  = val_df[val_df['forged'] == 1].iloc[0]

def predict_single(img_name, split, true_label):
    img_path = f'{BASE_PATH}/{split}/{img_name}'
    image    = Image.open(img_path).convert('RGB')
    tensor   = test_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)
        pred   = torch.argmax(output, dim=1).item()

    label_map = {0: 'genuine', 1: 'forged'}
    print(f"Image:      {img_name}")
    print(f"True label: {label_map[true_label]}")
    print(f"Predicted:  {label_map[pred]}")
    print(f"Confidence: genuine={probs[0][0]:.4f}, forged={probs[0][1]:.4f}")
    print(f"Correct:    {'✅' if pred == true_label else '❌'}")
    print()

print("Genuine Sample")
predict_single(genuine_row['image'], 'val', 0)

print("Forged Sample")
predict_single(forged_row['image'], 'val', 1)

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        preds   = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())

print(classification_report(
    all_labels, all_preds,
    target_names=['genuine', 'forged']
))

In [46]:
%%writefile /content/app.py
import os
os.system("apt-get install -y tesseract-ocr")
os.system("pip install pytesseract -q")

import re
import json
import torch
import numpy as np
from PIL import Image, ImageDraw
from torchvision import transforms, models
from transformers import LayoutLMForTokenClassification, BertTokenizerFast
from huggingface_hub import hf_hub_download
import gradio as gr

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS   = ["O", "B-VENDOR", "I-VENDOR", "B-DATE", "I-DATE", "B-TOTAL", "I-TOTAL"]
ID2LABEL = {i: x for i, x in enumerate(LABELS)}
LABEL2ID = {x: i for i, x in enumerate(LABELS)}

print("Loading models...")
tokenizer        = BertTokenizerFast.from_pretrained("microsoft/layoutlm-base-uncased")
extraction_model = LayoutLMForTokenClassification.from_pretrained(
    "Zakariya007/docfusion-v1",
    num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)
extraction_model = extraction_model.to(DEVICE)
extraction_model.eval()

forgery_model = models.efficientnet_b0(weights=None)
forgery_model.classifier[1] = torch.nn.Linear(1280, 2)
weights_path = hf_hub_download(
    repo_id="Zakariya007/docfusion-v2",
    filename="efficientnet_best.pth"
)
forgery_model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
forgery_model = forgery_model.to(DEVICE)
forgery_model.eval()
print("✅ Models loaded!")

def extract_fields(image):
    try:
        import pytesseract
        ocr_text = pytesseract.image_to_string(image)

        # Regex-based extraction
        date_match = re.search(
            r'\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}',
            ocr_text
        )
        date = date_match.group(0) if date_match else None

        total_match = re.search(
            r'(?:TOTAL|AMOUNT|JUMLAH)[^\d]*(\d+[\.,]\d{2})',
            ocr_text, re.IGNORECASE
        )
        total = total_match.group(1) if total_match else None

        lines  = [l.strip() for l in ocr_text.split('\n') if len(l.strip()) > 3]
        vendor = lines[0] if lines else None

        return vendor, date, total

    except Exception as e:
        print(f"Extraction error: {e}")
        return None, None, None


def detect_forgery(image, vendor, date, total):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output      = forgery_model(tensor)
        probs       = torch.softmax(output, dim=1)
        forged_prob = probs[0][1].item()
        visual_flag = forged_prob > 0.5

    rule_flags = []
    if not vendor: rule_flags.append("Missing vendor")
    if not date:   rule_flags.append("Missing date")
    if not total:  rule_flags.append("Missing total")

    if total:
        try:
            total_val = float(re.sub(r"[^\d.]", "", total))
            if total_val > 10000: rule_flags.append("Abnormally high total")
            if total_val <= 0:    rule_flags.append("Invalid total amount")
        except Exception:
            rule_flags.append("Unparseable total")

    if date:
        date_pattern = re.compile(
            r"\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}|\d{4}[\/\-\.]\d{2}[\/\-\.]\d{2}"
        )
        if not date_pattern.search(date):
            rule_flags.append("Invalid date format")

    rule_flag = len(rule_flags) >= 2
    is_forged = 1 if (visual_flag or rule_flag) else 0
    return is_forged, forged_prob, rule_flags


def annotate_image(image, is_forged):
    annotated = image.copy()
    draw      = ImageDraw.Draw(annotated)
    w, h      = image.size
    if is_forged:
        draw.rectangle([0, 0, w-1, h-1], outline="#FF0000", width=6)
        draw.text((10, 10), "FORGED", fill="#FF0000")
    else:
        draw.rectangle([0, 0, w-1, h-1], outline="#00AA00", width=6)
        draw.text((10, 10), "GENUINE", fill="#00AA00")
    return annotated


def process_receipt(image):
    if image is None:
        return None, "Please upload an image.", "", "", "", ""

    pil_image = Image.fromarray(image).convert("RGB")
    vendor, date, total        = extract_fields(pil_image)
    is_forged, forged_prob, rule_flags = detect_forgery(pil_image, vendor, date, total)
    annotated                  = annotate_image(pil_image, is_forged)

    if is_forged:
        status = f"FORGED (confidence: {forged_prob:.1%})"
        if rule_flags:
            status += "\nFlags: " + ", ".join(rule_flags)
    else:
        status = f"GENUINE (forged probability: {forged_prob:.1%})"

    return (
        np.array(annotated),
        status,
        vendor or "Not detected",
        date   or "Not detected",
        total  or "Not detected",
        json.dumps({
            "vendor":    vendor,
            "date":      date,
            "total":     total,
            "is_forged": is_forged
        }, indent=2),
    )


with gr.Blocks(title="DocFusion") as demo:
    gr.Markdown("# DocFusion — Intelligent Document Processing")
    gr.Markdown("Upload a scanned receipt to extract fields and detect forgery.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Upload Receipt", type="numpy")
            submit_btn  = gr.Button("Analyze Receipt", variant="primary")
        with gr.Column():
            output_image   = gr.Image(label="Annotated Receipt")
            forgery_status = gr.Textbox(label="Forgery Status", lines=4)

    with gr.Row():
        vendor_out = gr.Textbox(label="Vendor")
        date_out   = gr.Textbox(label="Date")
        total_out  = gr.Textbox(label="Total")

    json_out = gr.Code(label="JSON Output", language="json")

    submit_btn.click(
        fn      = process_receipt,
        inputs  = [input_image],
        outputs = [output_image, forgery_status, vendor_out, date_out, total_out, json_out],
    )

    gr.Markdown("**Models:** LayoutLM v1 (extraction) + EfficientNet-B0 (forgery)")

if __name__ == "__main__":
    demo.launch(share=True)

Overwriting /content/app.py


In [47]:
!python /content/app.py

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Loading models...
Loading weights: 100% 205/205 [00:00<00:00, 764.59it/s, Materializing param=layoutlm.pooler.dense.weight]
✅ Models loaded!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://df18d880f71731414e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://df18d880f71731414e.gradio.live


In [48]:
!git clone https://github.com/rihal-om/rihal-codestacker.git

Cloning into 'rihal-codestacker'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 294 (delta 10), reused 28 (delta 4), pack-reused 251 (from 2)
Receiving objects: 100% (294/294), 60.11 MiB | 14.18 MiB/s, done.
Resolving deltas: 100% (85/85), done.


In [49]:
import os
for item in os.listdir('/content/rihal-codestacker/ML'):
    print(item)

dummy_data
sample_submission
README.md
check_submission.py


In [52]:
!python /content/rihal-codestacker/ML/check_submission.py \
    --submission /content/drive/MyDrive/DocFusion \
    --data /content/rihal-codestacker/ML/dummy_data \
    --work-dir /content/tmp_work \
    --verbose

[check] Loaded submission: /content/drive/MyDrive/DocFusion
[train] Downloading extraction model (docfusion-v1)...
Loading weights: 100% 205/205 [00:00<00:00, 760.85it/s, Materializing param=layoutlm.pooler.dense.weight]
Writing model shards: 100% 1/1 [00:00<00:00,  2.37it/s]
[train] Extraction model saved.
[train] Downloading forgery model (docfusion-v2)...
[train] Forgery model saved.
[predict] Using device: cuda
Loading weights: 100% 205/205 [00:00<00:00, 782.95it/s, Materializing param=layoutlm.pooler.dense.weight]
[predict] Running inference on 10 records...
[predict] t001 → forged=0 vendor=ACME Corp date=2024-04-27 total=20.89
[predict] t002 → forged=0 vendor=City Store date=2024-05-03 total=109.44
[predict] t003 → forged=0 vendor=Quick Shop date=2024-12-11 total=110.25
[predict] t004 → forged=0 vendor=City Store date=2024-11-15 total=75.72
[predict] t005 → forged=0 vendor=Gulf Mart date=2024-04-24 total=282.88
[predict] t006 → forged=0 vendor=Quick Shop date=2024-07-19 total=202

In [53]:
import shutil

# Create submission zip
shutil.make_archive(
    '/content/docfusion_submission',
    'zip',
    '/content/drive/MyDrive/DocFusion'
)
print("✅ Submission zip created!")
print("📦 /content/docfusion_submission.zip")

✅ Submission zip created!
📦 /content/docfusion_submission.zip
